In [0]:
storage_account_name = "adlsretailmartabdul26"
storage_account_key = dbutils.secrets.get(scope="retailmart-kv-scope", key="adls-access-key")

# Configure Spark to use the account key for access
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
from pyspark.sql.functions import sum, count

# 1. Read clean Delta tables from Silver
df_orders = spark.read.format("delta").load(f"abfss://silver@{storage_account_name}.dfs.core.windows.net/orders")
df_items = spark.read.format("delta").load(f"abfss://silver@{storage_account_name}.dfs.core.windows.net/order_items")

# 2. Join Orders and Items
# We use a left join to ensure we keep all order items
df_fact_sales_raw = df_items.join(
    df_orders,
    on="order_id",
    how="left"
)

# 3. Select only the keys and metrics needed for the Fact table
df_fact_sales = df_fact_sales_raw.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_id",
    "order_purchase_timestamp", # This acts as our Date Key
    "price",
    "freight_value"
)

# 4. Write to Gold
gold_fact_sales_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/fact_sales"

df_fact_sales.write.format("delta") \
    .mode("overwrite") \
    .save(gold_fact_sales_path)

# 5. Preview the Fact Table
display(spark.read.format("delta").load(gold_fact_sales_path).limit(5))

order_id,order_item_id,product_id,seller_id,customer_id,order_purchase_timestamp,price,freight_value
00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2e45292979b9b2700ea09560eeb0f803,2017-10-16T15:29:43Z,21.33,15.1
00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2e45292979b9b2700ea09560eeb0f803,2017-10-16T15:29:43Z,21.33,15.1
00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2e45292979b9b2700ea09560eeb0f803,2017-10-16T15:29:43Z,21.33,15.1
00a379dfab816a83741012b71b264098,1,786248c98cab9ad5cccdb751909da1af,1da3aeb70d7989d1e6d9b0e887f97c23,5e3b89299a7d6e3e10f9bc6e77923f88,2018-08-03T19:50:50Z,15.99,7.39
01bc343311e5b8090b64d6ab287a5cf0,1,c8c969c3a6423f48df9fde8a13e6770b,b2479f944e1b90cf8a5de1bbfde284d6,3eef17cb7b0ffaf55b88ddfa3d793bfe,2018-02-22T14:30:12Z,21.99,15.1


In [0]:
from pyspark.sql.functions import col, initcap, trim

# 1. Read customer data from Bronze
bronze_customers_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/olist_customers_dataset.csv"

df_customers_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_customers_path)

# 2. Clean, deduplicate, and standardize city/state names
df_dim_customer = df_customers_raw \
    .filter(col("customer_id").isNotNull()) \
    .select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_zip_code_prefix"),
        initcap(trim(col("customer_city"))).alias("customer_city"), # Capitalize properly (e.g. "sao paulo" -> "Sao Paulo")
        trim(col("customer_state")).alias("customer_state")
    ) \
    .dropDuplicates(["customer_id"])

# 3. Write to Gold container
gold_customer_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/dim_customer"

df_dim_customer.write.format("delta") \
    .mode("overwrite") \
    .save(gold_customer_path)

# 4. Preview the Customer Dimension table
display(spark.read.format("delta").load(gold_customer_path).limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
001b4e9b73aa98a6276b241bfd496822,b6d2145b6b5de5dcc494c151af5fdce9,11436,Guaruja,SP
00b4054d851ec464f6fb745fab627439,82c199ce18d8d976f3029a4310e8ac5c,76825,Porto Velho,RO
0127d98d4a7e205081ddac271d6b8e52,c0f2dff1f913fe4237f13f4c98ecdd5a,3644,Sao Paulo,SP
015895fd0bf723100a64b0ead7c69cd6,ebbe2f9f5abdaaaaad1d0999207e9362,13454,Santa Barbara D'oeste,SP
01a56495d1c8bcf831311f5ad7f126a1,5733152cb1f37713b8dbe91142bcf91b,13905,Amparo,SP


In [0]:
# 1. Define paths for the products and the translation table in Bronze
bronze_products_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/olist_products_dataset.csv"
bronze_translation_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/product_category_name_translation.csv"

# 2. Read both files
df_products_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(bronze_products_path)
df_translation = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(bronze_translation_path)

# 3. Join them to get the English name, filter nulls, and select the needed columns
df_dim_product = df_products_raw \
    .filter(col("product_id").isNotNull()) \
    .join(df_translation, on="product_category_name", how="left") \
    .select(
        col("product_id"),
        col("product_category_name_english").alias("product_category"), # Use the English column
        col("product_weight_g"),
        col("product_length_cm"),
        col("product_height_cm"),
        col("product_width_cm")
    ) \
    .dropDuplicates(["product_id"])

# 4. Write to Gold container
gold_product_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/dim_product"

df_dim_product.write.format("delta") \
    .mode("overwrite") \
    .save(gold_product_path)

# 5. Preview the Product Dimension table
display(spark.read.format("delta").load(gold_product_path).limit(5))

product_id,product_category,product_weight_g,product_length_cm,product_height_cm,product_width_cm
002159fe700ed3521f46cfcf6e941c76,fashion_shoes,1850,36,37,16
0021a87d4997a48b6cef1665602be0f5,computers_accessories,650,25,10,15
00bc6e6d4ceb98ec7ed058b0a243a8b9,christmas_supplies,200,16,11,11
0110986ff84d644fee920efb18577292,auto,2900,72,4,44
0152f69b6cf919bcdaf117aa8c43e5a2,bed_bath_table,250,40,4,30


In [0]:
# 1. Define the path for sellers in Bronze
bronze_sellers_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/olist_sellers_dataset.csv"

# 2. Read the raw file
df_sellers_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_sellers_path)

# 3. Clean, deduplicate, and standardize city/state names
df_dim_seller = df_sellers_raw \
    .filter(col("seller_id").isNotNull()) \
    .select(
        col("seller_id"),
        col("seller_zip_code_prefix"),
        initcap(trim(col("seller_city"))).alias("seller_city"), # Capitalize properly
        trim(col("seller_state")).alias("seller_state")
    ) \
    .dropDuplicates(["seller_id"])

# 4. Write to Gold container
gold_seller_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/dim_seller"

df_dim_seller.write.format("delta") \
    .mode("overwrite") \
    .save(gold_seller_path)

# 5. Preview the Seller Dimension table
display(spark.read.format("delta").load(gold_seller_path).limit(5))

seller_id,seller_zip_code_prefix,seller_city,seller_state
001cca7ae9ae17fb1caed9dfb1094831,29156,Cariacica,ES
002100f778ceb8431b7a1020ff7ab48f,14405,Franca,SP
004c9cd9d87a3c30c522c48c4fc07416,14940,Ibitinga,SP
00ee68308b45bc5e2660cd833c3f81cc,3333,Sao Paulo,SP
0176f73cc1195f367f7b32db1e5b3aa8,14940,Ibitinga,SP


In [0]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, quarter, dayofweek, date_format, col

# 1. Extract every unique date from our existing fact_sales table
# We use to_date() to strip the time (hours/minutes) and just keep the day
df_unique_dates = df_fact_sales \
    .select(to_date(col("order_purchase_timestamp")).alias("date_key")) \
    .distinct() \
    .filter(col("date_key").isNotNull())

# 2. Add all the calendar attributes
df_dim_date = df_unique_dates \
    .withColumn("year", year("date_key")) \
    .withColumn("month", month("date_key")) \
    .withColumn("day", dayofmonth("date_key")) \
    .withColumn("quarter", quarter("date_key")) \
    .withColumn("day_of_week_num", dayofweek("date_key")) \
    .withColumn("day_of_week_name", date_format("date_key", "EEEE")) # EEEE gives the full name, e.g., 'Monday'

# 3. Write to Gold container
gold_date_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/dim_date"

df_dim_date.write.format("delta") \
    .mode("overwrite") \
    .save(gold_date_path)

# 4. Preview the Date Dimension table
display(spark.read.format("delta").load(gold_date_path).limit(5))

date_key,year,month,day,quarter,day_of_week_num,day_of_week_name
2018-08-10,2018,8,10,3,6,Friday
2018-05-28,2018,5,28,2,2,Monday
2017-09-11,2017,9,11,3,2,Monday
2017-08-11,2017,8,11,3,6,Friday
2018-06-06,2018,6,6,2,4,Wednesday


In [0]:
# 1. Register the Delta tables as temporary SQL views 
spark.read.format("delta").load(gold_fact_sales_path).createOrReplaceTempView("fact_sales")
spark.read.format("delta").load(gold_product_path).createOrReplaceTempView("dim_product")

# 2. Write the SQL query to join the Fact and Dimension tables
query = """
SELECT 
    p.product_category,
    ROUND(SUM(f.price), 2) AS total_revenue,
    COUNT(f.order_id) AS total_items_sold
FROM fact_sales f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.product_category
ORDER BY total_revenue DESC
LIMIT 5
"""

# 3. Run and display the result
display(spark.sql(query))

product_category,total_revenue,total_items_sold
watches_gifts,13136.21,65
health_beauty,11869.72,91
bed_bath_table,10901.08,120
furniture_decor,8444.27,93
computers_accessories,7308.34,64
